In [ ]:
def extract_network_features(self, dict_adj_matrix: dict) -> (pd.DataFrame, pd.DataFrame):
    """
    Извлекает узловые фичи сети из словаря матриц смежности.

    Для каждой симуляции:
      - Строится граф из матрицы смежности.
      - Вычисляются:
          * Degree centrality
          * Betweenness centrality
          * Closeness centrality
          * Eigenvector centrality
          * Local clustering coefficient
          * PageRank
          * Average neighbor degree
          * k-Core number
          * Eccentricity
          * Local efficiency (локальная эффективность)
      - Выполняется разбиение на сообщества (алгоритм Girvan–Newman, первая итерация).
    
    Возвращает:
      - df_features: DataFrame с узловыми фичами (n_sim, node, и прочие метрики).
      - df_communities: DataFrame с информацией о принадлежности к сообществу (n_sim, node, community).
    """
    import networkx as nx
    import numpy as np
    import pandas as pd
    from tqdm import tqdm

    def local_efficiency_node(G, node):
        """
        Вычисляет локальную эффективность для узла: 
        эффективность подграфа, индуцированного соседями данного узла.
        Если соседей меньше 2, возвращает 0.
        """
        neighbors = list(G.neighbors(node))
        if len(neighbors) < 2:
            return 0.0
        H = G.subgraph(neighbors)
        total, count = 0.0, 0
        for u in neighbors:
            lengths = nx.single_source_shortest_path_length(H, u)
            for v in neighbors:
                if u != v:
                    if v in lengths and lengths[v] > 0:
                        total += 1 / lengths[v]
                    # Если узлы не связаны, вклад считается 0.
                    count += 1
        return total / count if count > 0 else 0.0

    features_list = []
    community_list = []

    for sim, adj_matrix in tqdm(dict_adj_matrix.items(), desc="Extracting network features"):
        G = nx.from_numpy_array(adj_matrix)
        # Основные центральности:
        deg_cent = nx.degree_centrality(G)
        bet_cent = nx.betweenness_centrality(G)
        close_cent = nx.closeness_centrality(G)
        try:
            eig_cent = nx.eigenvector_centrality_numpy(G)
        except Exception:
            eig_cent = {node: np.nan for node in G.nodes()}
        # Локальный коэффициент кластеризации
        clust_coeff = nx.clustering(G)
        # PageRank
        pr = nx.pagerank(G)
        # Средняя степень соседей
        avg_neigh_deg = nx.average_neighbor_degree(G)
        # k-Core number
        try:
            k_core = nx.core_number(G)
        except Exception:
            k_core = {node: np.nan for node in G.nodes()}
        # Eccentricity (если граф не связный, вычисляем для каждого компонента)
        try:
            ecc = nx.eccentricity(G)
        except nx.NetworkXError:
            ecc = {}
            for component in nx.connected_components(G):
                subG = G.subgraph(component)
                sub_ecc = nx.eccentricity(subG)
                ecc.update(sub_ecc)
        # Локальная эффективность для каждого узла
        local_eff = {node: local_efficiency_node(G, node) for node in G.nodes()}

        # Определение сообществ (первая итерация алгоритма Girvan-Newman)
        communities_generator = nx.algorithms.community.girvan_newman(G)
        try:
            first_partition = next(communities_generator)
        except StopIteration:
            first_partition = [set(G.nodes())]
        node_comm = {}
        for idx, community in enumerate(first_partition):
            for node in community:
                node_comm[node] = idx

        for node in G.nodes():
            features_list.append({
                "n_sim": sim,
                "node": node,
                "degree_centrality": deg_cent[node],
                "betweenness_centrality": bet_cent[node],
                "closeness_centrality": close_cent[node],
                "eigenvector_centrality": eig_cent[node],
                "clustering_coefficient": clust_coeff[node],
                "pagerank": pr[node],
                "avg_neighbor_degree": avg_neigh_deg[node],
                "k_core": k_core[node],
                "eccentricity": ecc[node],
                "local_efficiency": local_eff[node]
            })
            community_list.append({
                "n_sim": sim,
                "node": node,
                "community": node_comm.get(node, -1)
            })

    df_features = pd.DataFrame(features_list)
    df_communities = pd.DataFrame(community_list)
    return df_features, df_communities
